# Day 12: Quantization — GPTQ, AWQ, SmoothQuant
> *Inference Engineering* — Chapter 5.1.2 | Philip Kiely, Baseten Books 2026

**Layer:** Runtime | **Prerequisite:** Day 11 (Number Formats)


## Concept Overview

Post-training quantization (PTQ) algorithms minimize accuracy loss when quantizing to INT4/INT8. GPTQ uses second-order weight correction via Hessian information to compensate for rounding errors after quantizing each weight. AWQ (Activation-aware Weight Quantization) identifies the ~1% of channels with large activations and scales them before quantization, protecting the most important weights. SmoothQuant transfers quantization difficulty from activations (hard to quantize) to weights (easy to quantize) by migrating scaling factors between the two. All three achieve near-FP16 accuracy at INT4 weight precision.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. The Problem: Activation Outliers

LLM activations have extreme outliers in certain channels — values 100x larger than typical. These outliers dominate the quantization range, causing most values to map to just a few quantization bins.


In [ ]:
# Simulate activation outlier problem
np.random.seed(42)
torch.manual_seed(42)

# Typical LLM hidden state: most channels normal, a few with large outliers
d_model = 4096
activations = torch.randn(32, d_model) * 0.5  # 32 tokens, 4096 channels

# Inject outlier channels (1% of channels have 100x larger values)
outlier_channels = torch.randperm(d_model)[:int(d_model*0.01)]
activations[:, outlier_channels] *= 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Channel max values
channel_max = activations.abs().max(dim=0).values
ax1.semilogy(sorted(channel_max.numpy(), reverse=True))
ax1.set_xlabel('Channel (sorted by magnitude)')
ax1.set_ylabel('Max |activation|')
ax1.set_title('Activation Magnitude Distribution')
ax1.axvline(int(d_model*0.01), color='red', linestyle='--', label='Outlier threshold')
ax1.legend()
ax1.grid(True)

# Show per-tensor vs per-channel quantization quality
def quant_int8(x, per_channel=False):
    if per_channel:
        scale = x.abs().max(dim=-1, keepdim=True).values / 127
    else:
        scale = x.abs().max() / 127
    scale = scale.clamp(min=1e-8)
    return (x / scale).round().clamp(-127, 127) * scale

q_tensor = quant_int8(activations, per_channel=False)
q_channel = quant_int8(activations, per_channel=True)

err_tensor = (activations - q_tensor).abs().mean().item()
err_channel = (activations - q_channel).abs().mean().item()

ax2.bar(['Per-tensor INT8', 'Per-channel INT8'], [err_tensor, err_channel],
        color=['salmon','steelblue'])
ax2.set_ylabel('Mean Absolute Error')
ax2.set_title('Quantization Error: Tensor vs Channel')
for i, v in enumerate([err_tensor, err_channel]):
    ax2.text(i, v, f'{v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('activation_outliers.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Per-tensor INT8 MAE:  {err_tensor:.4f}')
print(f'Per-channel INT8 MAE: {err_channel:.4f}')
print(f'Per-channel improvement: {err_tensor/err_channel:.1f}x')


## 2. SmoothQuant — Migrating Quantization Difficulty

SmoothQuant scales activations down (by $s$) and weights up (by $s$) along the channel dimension, moving the quantization difficulty from activations to weights. Since weights can be quantized offline (per-channel), this dramatically reduces activation quantization error.

$$\hat{Y} = (X \text{diag}(s)^{-1}) \cdot (\text{diag}(s) W) = \hat{X} \hat{W}$$


In [ ]:
def smooth_quant_scale(activations, weights, alpha=0.5):
    """
    Compute per-channel smoothing scale for SmoothQuant.
    alpha controls how much difficulty shifts to weights (0=all to activations, 1=all to weights)
    """
    # Channel-wise max stats
    act_max = activations.abs().max(dim=0).values  # [d_model]
    w_max = weights.abs().max(dim=0).values         # [d_model] (per output channel)

    scale = (act_max**alpha) / (w_max**(1-alpha) + 1e-8)
    return scale

# Simulate SmoothQuant on a linear layer
torch.manual_seed(42)
batch, d_in, d_out = 32, 512, 1024

X = torch.randn(batch, d_in) * 0.5
X[:, :5] *= 100  # 5 outlier channels
W = torch.randn(d_out, d_in) * 0.02

# Without SmoothQuant
def quant_tensor(x, bits=8):
    scale = x.abs().max() / (2**(bits-1)-1)
    return (x/scale).round().clamp(-(2**(bits-1)-1), 2**(bits-1)-1) * scale

X_q = quant_tensor(X)
W_q = quant_tensor(W)
err_naive = ((X @ W.T) - (X_q @ W_q.T)).abs().mean().item()

# With SmoothQuant
scales = smooth_quant_scale(X, W.T)
X_smooth = X / scales
W_smooth = W * scales
X_smooth_q = quant_tensor(X_smooth)
W_smooth_q = quant_tensor(W_smooth)
err_smooth = ((X @ W.T) - (X_smooth_q @ W_smooth_q.T)).abs().mean().item()

print(f'Naive INT8 quantization error:     {err_naive:.4f}')
print(f'SmoothQuant INT8 quantization error: {err_smooth:.4f}')
print(f'Improvement: {err_naive/err_smooth:.1f}x lower error')

# Show scale distribution
plt.figure(figsize=(8, 4))
plt.hist(scales.numpy(), bins=50)
plt.xlabel('Smoothing Scale')
plt.ylabel('Count')
plt.title('SmoothQuant Per-Channel Scales')
plt.savefig('smoothquant_scales.png', dpi=100, bbox_inches='tight')
plt.show()


## 3. AWQ — Activation-Aware Weight Quantization

AWQ finds the ~1% of weight channels with the largest activation magnitudes and applies a per-channel scale to protect them. Unlike SmoothQuant, AWQ is weight-only quantization — activations remain in FP16.


In [ ]:
def awq_scale(activations, weights, top_frac=0.01, grid_size=20):
    """
    Simplified AWQ: search for per-channel scale that minimizes quantization error
    for the most salient channels.
    """
    act_importance = activations.abs().mean(dim=0)  # [d_in]

    # Identify salient channels
    n_salient = max(1, int(len(act_importance) * top_frac))
    salient_channels = act_importance.topk(n_salient).indices

    # Grid search for best scale
    best_scale = torch.ones(weights.shape[1])
    best_err = float('inf')
    true_out = X @ W.T

    for s_exp in np.linspace(-1, 1, grid_size):
        scale = torch.ones(weights.shape[1])
        scale[salient_channels] = 2**s_exp
        W_scaled = W * scale
        W_q = quant_tensor(W_scaled.contiguous())
        W_dq = W_q / scale
        out_q = X @ W_dq.T
        err = (true_out - out_q).abs().mean().item()
        if err < best_err:
            best_err = err
            best_scale = scale.clone()

    return best_scale, best_err

best_scale, awq_err = awq_scale(X, W)
print(f'AWQ INT4-style quantization error: {awq_err:.4f}')
print(f'Improvement vs naive: {err_naive/awq_err:.1f}x')
print(f'AWQ salient channel scale range: [{best_scale.min():.3f}, {best_scale.max():.3f}]')

print('\nSummary comparison:')
for method, err in [('Naive INT8', err_naive), ('SmoothQuant', err_smooth), ('AWQ-style', awq_err)]:
    print(f'  {method:<20} error={err:.4f}')


## Experiments: Try These

1. **Alpha sensitivity in SmoothQuant**: Sweep alpha from 0 to 1 and plot quantization error. What alpha minimizes error for your weight/activation distribution?
2. **Block-wise GPTQ**: Implement Optimal Brain Quantization (OBQ) — quantize weights one at a time, correcting remaining weights using the inverse Hessian.
3. **Perplexity measurement**: Quantize a small GPT-2 model (117M) with naive INT8 vs AWQ-style scaling and measure perplexity on WikiText-2.


## Key Takeaways

- LLM activations have ~1% of channels with 100x outlier magnitudes — these ruin naive per-tensor quantization.
- SmoothQuant migrates difficulty from activations (hard) to weights (easy) by applying channel-wise scales, enabling efficient INT8 W8A8 quantization.
- AWQ identifies salient activation channels and applies per-channel scales to weights, achieving near-FP16 accuracy at INT4 weight precision.
- GPTQ uses second-order Hessian information to compensate for rounding errors — more compute at quantization time, better accuracy at INT4.

## References
- *Inference Engineering* Ch 5.1.2 — Philip Kiely, Baseten Books 2026
- Xiao et al. (2022), "SmoothQuant"
- Lin et al. (2023), "AWQ: Activation-aware Weight Quantization"
- Frantar et al. (2022), "GPTQ: Accurate Post-Training Quantization"
